# 3. Politeness Feature Construction

*AI assistance (Claude-sonnet-4.6 in UvA AI Chat): notebook structure and ConvoKit pipeline setup.*

This notebook takes `messages.csv` produced by notebook 02 and extracts
politeness strategy scores for each message using ConvoKit's
`PolitenessStrategies` transformer (Danescu-Niculescu-Mizil et al., 2013).
It runs four steps in order:

| Step | Input | Output |
|------|-------|--------|
| 3.1 | `messages.csv` | Validated message dataframe |
| 3.2 | Message texts (via MediaWiki API) | `messages_with_text.csv` |
| 3.3 | `messages_with_text.csv` + ConvoKit | `politeness_scores.csv` |
| 3.4 | `politeness_scores.csv` | `politeness_profiles.csv` (newcomer-level) |

**Note:** `ConvoKit` is incompatible with Python 3.11 and above.

### 3.0 Configuration

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
import convokit
from convokit import Corpus, Speaker, Utterance, PolitenessStrategies
from convokit.politenessStrategies import PolitenessStrategies
import spacy
from convokit import TextParser

In [ ]:
# ── Working directory ────────────────────────────────────────────────────────
os.chdir("/Users/ada/bdacaII-2026-ada")

# ── API endpoint ──────────────────────────────────────────────────────────────
API = "https://en.wikipedia.org/w/api.php"

# ── Polite API call settings (must match notebooks 1 and 2) ──────────────────
MAXLAG              = 5    # seconds; cooperative DB-lag threshold
REQUEST_TIMEOUT     = 15   # seconds before a request is considered hung
RETRY_DELAY         = 2    # seconds to wait between retry attempts
MAX_RETRIES         = 5    # maximum retry attempts per API call
INTER_REQUEST_DELAY = 0.1  # polite pause between individual API calls (in seconds)
INTER_PAGE_DELAY    = 0.5  # polite pause between paginated API calls (in seconds)
SAMPLE_CAP = 1000          # max messages per sender type
SEED       = 42
BATCH_SIZE = 50            # MediaWiki API maximum revids per request
SAVE_EVERY = 20            # save progress every N batches

# ── Data directories (must match notebooks 1 and 2) ──────────────────────────
DATA_DIR      = Path("Final_project/data")
PROCESSED_DIR = DATA_DIR / "processed"

# ── Input paths ───────────────────────────────────────────────────────────────
MESSAGES_CSV  = PROCESSED_DIR / "messages.csv"

# ── Output paths ──────────────────────────────────────────────────────────────
MESSAGES_TEXT_CSV     = PROCESSED_DIR / "messages_with_text.csv"
POLITENESS_SCORES_CSV = PROCESSED_DIR / "politeness_scores.csv"
POLITENESS_PROFILES_CSV = PROCESSED_DIR / "politeness_profiles.csv"

# ── Module-level session (same User-Agent as notebooks 1 and 2) ──────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "BDACA-II-course/1.0 (educational; adashiyj@gmail.com)"
})

print(f"pandas     {pd.__version__}")
print(f"Session ready — User-Agent: {SESSION.headers['User-Agent']}")

### 3.1 Load, validate, and subsample message dataframe

Load messages.csv from notebook 02, validate columns, exclude unidentified bots, and subsample to SAMPLE_CAP messages per sender type before fetching any text. <br>This keeps the fetch and scoring steps more computationally feasible.

In [ ]:
def load_and_subsample(
    path: Path = MESSAGES_CSV,
    cap: int   = SAMPLE_CAP,
    seed: int  = SEED,
) -> pd.DataFrame:
    """Load, validate, and subsample the message-level dataframe.

    Parameters
    ----------
    path : Path to messages.csv produced by notebook 02.
    cap  : Maximum messages to retain per sender type. Sender types with
           fewer messages than cap are kept in full.
    seed : Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame with columns: user_id, username, revid, timestamp,
    sender, sender_type, sizediff.

    Raises
    ------
    FileNotFoundError  If messages.csv does not exist.
    ValueError         If expected columns are missing.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"messages.csv not found at {path}.\n"
            "Run notebook 02 first to generate it."
        )

    df = pd.read_csv(path)

    expected_cols = {"user_id", "username", "revid", "sender_type"}
    missing = expected_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    # ── Exclude unidentified bots ─────────────────────────────────────────────
    n_before  = len(df)
    df        = df[df["sender_type"] != "unidentified_bot"].copy()
    n_excl    = n_before - len(df)

    print(f"Messages loaded        : {n_before:,}")
    print(f"Unidentified bots excl : {n_excl:,}")
    print(f"Messages for scoring   : {len(df):,}")
    print(f"\nSender type counts (before subsampling):")
    print(df["sender_type"].value_counts().to_string())

    # ── Subsample ─────────────────────────────────────────────────────────────
    df_sampled = (
        df
        .groupby("sender_type", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), cap), random_state=seed))
        .reset_index(drop=True)
    )

    print(f"\nSender type counts (after subsampling to {cap:,} per type):")
    print(df_sampled["sender_type"].value_counts().to_string())
    print(f"\nTotal messages to fetch and score: {len(df_sampled):,}")

    return df_sampled


df_messages = load_and_subsample()
df_messages.head()

### 3.2 Fetch message text from the MediaWiki API (batched)

Retrieve the wikitext content of each revision using the `revisions`
endpoint with `rvprop=content`. 
<br>Revisions are fetched in batches of 50 (the API maximum) to reduce total request count from individual calls to batched calls. 
<br>Progress is saved every `SAVE_EVERY` batches so the step can be resumed if interrupted.

In [ ]:
def fetch_batch_texts(revids: list[int]) -> dict[int, str]:
    """Fetch wikitext for a batch of up to 50 revision IDs in one API call.

    Parameters
    ----------
    revids : List of revision IDs (max 50).

    Returns
    -------
    Dict mapping revid -> wikitext content string (empty string on failure).
    """
    params = {
        "action":        "query",
        "prop":          "revisions",
        "revids":        "|".join(str(r) for r in revids),
        "rvprop":        "content",
        "rvslots":       "main",
        "format":        "json",
        "formatversion": "2",
        "maxlag":        str(MAXLAG),
    }

    for attempt in range(MAX_RETRIES):
        try:
            resp  = SESSION.get(API, params=params, timeout=REQUEST_TIMEOUT)
            data  = resp.json()

            # Honour maxlag retry signal from the server.
            if "error" in data and data["error"].get("code") == "maxlag":
                wait = int(resp.headers.get("Retry-After", RETRY_DELAY))
                time.sleep(wait)
                continue

            pages = data.get("query", {}).get("pages", [])
            result = {}
            for page in pages:
                for rev in page.get("revisions", []):
                    rid     = rev.get("revid")
                    content = rev.get("slots", {}).get("main", {}).get("content", "")
                    if rid is not None:
                        result[rid] = content
            return result

        except Exception:
            time.sleep(RETRY_DELAY * (attempt + 1))

    # All retries exhausted — return empty strings for the whole batch.
    return {r: "" for r in revids}


def collect_message_texts(
    df: pd.DataFrame,
    output_path: Path = MESSAGES_TEXT_CSV,
    batch_size: int   = BATCH_SIZE,
    save_every: int   = SAVE_EVERY,
) -> pd.DataFrame:
    """Fetch revision text for all messages in df using batched API calls.

    Skips revisions already fetched if output_path exists (crash-safe resume).

    Parameters
    ----------
    df          : Subsampled message dataframe from load_and_subsample().
    output_path : CSV path for incremental saves.
    batch_size  : Number of revids per API request (max 50).
    save_every  : Save progress every N batches.

    Returns
    -------
    df with a new column `text` containing the revision wikitext.
    """
    # ── Resume from prior run if available ───────────────────────────────────
    if output_path.exists():
        df_saved     = pd.read_csv(output_path)
        already_done = set(df_saved["revid"])
        print(f"Resuming: {len(already_done):,} revisions already fetched.")
    else:
        df_saved     = pd.DataFrame(columns=["revid", "text"])
        already_done = set()

    to_fetch = df[~df["revid"].isin(already_done)]["revid"].tolist()
    n_total  = len(to_fetch)
    n_batches = (n_total + batch_size - 1) // batch_size
    print(f"Fetching {n_total:,} revisions in {n_batches:,} batches of {batch_size}.")

    results = []
    for batch_idx in range(n_batches):
        batch   = to_fetch[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        texts   = fetch_batch_texts(batch)
        for revid in batch:
            results.append({"revid": revid, "text": texts.get(revid, "")})

        # ── Incremental save ──────────────────────────────────────────────────
        if (batch_idx + 1) % save_every == 0:
            df_partial  = pd.DataFrame(results)
            df_combined = pd.concat([df_saved, df_partial], ignore_index=True)
            df_combined.to_csv(output_path, index=False)
            pct = (batch_idx + 1) / n_batches * 100
            print(f"  Batch {batch_idx + 1}/{n_batches} ({pct:.0f}%) — saved.")

        time.sleep(INTER_REQUEST_DELAY)  # polite pause between batches

    # ── Final save ────────────────────────────────────────────────────────────
    df_new = pd.DataFrame(results)
    df_all = pd.concat([df_saved, df_new], ignore_index=True).drop_duplicates("revid")
    df_all.to_csv(output_path, index=False)

    # ── Merge text back onto message dataframe ────────────────────────────────
    df_with_text         = df.merge(df_all[["revid", "text"]], on="revid", how="left")
    df_with_text["text"] = df_with_text["text"].fillna("")

    empty = (df_with_text["text"] == "").sum()
    print(f"\nFetch complete.")
    print(f"  Total messages : {len(df_with_text):,}")
    print(f"  Empty text     : {empty:,} ({empty / len(df_with_text):.1%})")

    return df_with_text


df_messages = collect_message_texts(df_messages)
df_messages.head()

### 3.3 Compute politeness strategies with ConvoKit

Pass each message through ConvoKit's `PolitenessStrategies` transformer
(Danescu-Niculescu-Mizil et al., 2013). This yields 20 binary strategy
scores per message. Messages with empty text are assigned zero scores
for all strategies and flagged with `scored=False`.

In [ ]:
def build_convokit_corpus(df: pd.DataFrame) -> Corpus:
    """Convert the message dataframe into a ConvoKit Corpus.

    Only non-empty messages are added; empty-text rows are handled
    separately after scoring.

    Parameters
    ----------
    df : Message dataframe with columns revid, sender, text.

    Returns
    -------
    ConvoKit Corpus ready for TextParser + PolitenessStrategies.
    """
    df_nonempty = df[df["text"].str.strip() != ""].copy()

    speakers = {
        str(sender): Speaker(id=str(sender))
        for sender in df_nonempty["sender"].unique()
    }

    utterances = [
        Utterance(
            id=str(row.revid),
            speaker=speakers[str(row.sender)],
            conversation_id="all_messages",
            text=str(row.text),
        )
        for row in df_nonempty.itertuples(index=False)
    ]

    print(f"Corpus built: {len(utterances):,} utterances, "
          f"{len(speakers):,} unique speakers.")
    return Corpus(utterances=utterances)


def score_politeness(
    df: pd.DataFrame,
    output_path: Path = POLITENESS_SCORES_CSV,
) -> pd.DataFrame:
    """Extract politeness strategy scores for all messages.

    Pipeline:
      1. Build ConvoKit Corpus from non-empty messages.
      2. Parse utterances with TextParser (spaCy dependency parse).
      3. Score with PolitenessStrategies → 21 binary strategy columns.
      4. Append zero-score rows for empty/NaN-text messages (scored=False).

    Parameters
    ----------
    df          : Message dataframe with text column (from Step 3.2).
    output_path : CSV path to save scores to.

    Returns
    -------
    pd.DataFrame with revid, user_id, sender_type, scored,
    and 21 politeness strategy columns.
    """
    if output_path.exists():
        print(f"Scores already exist at {output_path} — loading.")
        return pd.read_csv(output_path)

    # ── Step 1: build corpus ──────────────────────────────────────────────────
    print("Building ConvoKit corpus ...")
    corpus = build_convokit_corpus(df)

    # ── Step 2: parse with spaCy ──────────────────────────────────────────────
    print("Parsing utterances with TextParser (spaCy) ...")
    parser = TextParser(verbosity=100)
    corpus = parser.transform(corpus)

    # ── Step 3: score politeness strategies ───────────────────────────────────
    print("Scoring politeness strategies ...")
    ps     = PolitenessStrategies()
    corpus = ps.transform(corpus, markers=True)

    # ── Step 4: extract scores — defensive version ────────────────────────────
    rows = []
    for utt in corpus.iter_utterances():
        scores = utt.retrieve_meta("politeness_strategies")
        if scores is None:
            continue
        # utt.id was set to str(revid); cast back to int safely
        try:
            revid = int(utt.id)
        except (ValueError, TypeError):
            print(f"  Warning: could not parse utt.id={utt.id!r} as int — skipping.")
            continue
        row = {"revid": revid}
        row.update(scores)
        rows.append(row)

    print(f"Utterances with scores extracted: {len(rows):,}")

    # If nothing was extracted, print a sample of utt IDs to diagnose
    if len(rows) == 0:
        sample_ids = [utt.id for utt in list(corpus.iter_utterances())[:5]]
        print(f"No rows extracted. Sample utt.id values: {sample_ids}")
        raise RuntimeError(
            "No politeness scores were extracted from the corpus. "
            "Check the utt.id values printed above."
        )

    df_scores = pd.DataFrame(rows)
    print(f"df_scores columns: {df_scores.columns.tolist()}")  # diagnostic

    # ── Step 5: merge user_id and sender_type back in ─────────────────────────
    # Use revid from df; drop duplicates so merge is clean
    df_meta = df[["revid", "user_id", "sender_type"]].drop_duplicates(subset="revid")
    df_scores = df_scores.merge(df_meta, on="revid", how="left")
    df_scores["scored"] = True

    # ── Step 6: append zero-score rows for missing-text messages ─────────────
    strategy_cols = [
        c for c in df_scores.columns
        if c not in {"revid", "user_id", "sender_type", "scored"}
    ]
    empty_rows = df[df["text"].isna() | (df["text"].str.strip() == "")][
        ["revid", "user_id", "sender_type"]
    ].copy()

    if not empty_rows.empty:
        for col in strategy_cols:
            empty_rows[col] = 0
        empty_rows["scored"] = False
        df_scores = pd.concat([df_scores, empty_rows], ignore_index=True)
        print(f"  Empty/NaN-text messages (zero-scored): {len(empty_rows):,}")

    df_scores.to_csv(output_path, index=False)

    n_scored = df_scores["scored"].sum()
    print(f"\nPoliteness scoring complete.")
    print(f"  Messages scored      : {n_scored:,}")
    print(f"  Messages zero-scored : {(~df_scores['scored']).sum():,}")
    print(f"  Strategies extracted : {len(strategy_cols)}")

    return df_scores


df_scores = score_politeness(df_messages)
df_scores.head()

### 3.4 Aggregate politeness scores to newcomer level

Average the 20 strategy scores per sender type per newcomer.
This produces the RQ2 predictor variables: one average score
per strategy per sender type at the newcomer level.

In [ ]:
def build_politeness_profiles(
    df_scores: pd.DataFrame,
    output_path: Path = POLITENESS_PROFILES_CSV,
) -> pd.DataFrame:
    """Aggregate message-level politeness scores to the newcomer level.

    For each newcomer and sender type, computes the mean score for each
    of the 20 politeness strategies across all messages received from
    that sender type. Column names follow the pattern:
    {sender_type}__{strategy_name}.

    Parameters
    ----------
    df_scores   : Message-level politeness scores (from Step 3.3).
    output_path : CSV path to save the newcomer-level profiles to.

    Returns
    -------
    pd.DataFrame with one row per newcomer and aggregated strategy columns.
    """
    strategy_cols = [c for c in df_scores.columns
                     if c not in {"revid", "user_id", "sender_type", "scored"}]

    # Only use successfully scored messages for averaging.
    df_scored = df_scores[df_scores["scored"] == True].copy()

    sender_types = df_scored["sender_type"].unique()
    profile_dfs  = []

    for sender_type in sender_types:
        df_sender = df_scored[df_scored["sender_type"] == sender_type]

        # Average strategy scores per newcomer for this sender type.
        df_agg = (
            df_sender.groupby("user_id")[strategy_cols]
            .mean()
            .reset_index()
        )

        # Prefix column names with sender type for clarity.
        rename_map = {
            col: f"{sender_type}__{col}" for col in strategy_cols
        }
        df_agg = df_agg.rename(columns=rename_map)
        profile_dfs.append(df_agg)

    # Merge all sender-type profiles on user_id.
    df_profiles = profile_dfs[0]
    for df_next in profile_dfs[1:]:
        df_profiles = df_profiles.merge(df_next, on="user_id", how="outer")

    # Fill NaN with 0 — newcomer received no messages from that sender type.
    df_profiles = df_profiles.fillna(0)

    df_profiles.to_csv(output_path, index=False)
    print(f"Politeness profiles: {len(df_profiles):,} newcomers x "
          f"{len(df_profiles.columns)} columns")
    print(f"Saved to {output_path}")

    return df_profiles


df_profiles = build_politeness_profiles(df_scores)
df_profiles.head()

### 3.5 Summary

Sanity check of all output files before proceeding to analysis.

In [ ]:
print("=" * 55)
print("Politeness feature construction complete")
print("=" * 55)

for path in (MESSAGES_TEXT_CSV, POLITENESS_SCORES_CSV, POLITENESS_PROFILES_CSV):
    if path.exists():
        df_check = pd.read_csv(path)
        print(f"  {path.name:40s} {len(df_check):>6,} rows x {len(df_check.columns)} cols")
    else:
        print(f"  {path.name:40s} NOT FOUND")

print()